# Week 3a: compress the fine-tuned model and measure what it costs in quality

**Kaggle settings:** Accelerator = *GPU T4 x2*, Internet = *on*. **Secrets** (optional): `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`.

`python scripts/kaggle_run.py run --week 3 --part a --attach models/week1_run3/sft/adapter=qwen-sql-week1-adapter`

What runs: merge the Week 1 adapter (fp16), quantize it with llm-compressor (AWQ W4A16, GPTQ W4A16, SmoothQuant+GPTQ W8A8-INT8; calibration = 256 Spider *train* rows), export GGUF Q8_0 / Q4_K_M with llama.cpp, then for every variant: Spider dev execution accuracy, WikiText-2 and Spider-SQL perplexity, flip matrix + McNemar vs fp16. Last, the Q4_K_M GGUF on **llama.cpp CPU** (the cloud engine) on 200 dev questions.

The models stay in `/kaggle/working/models` as kernel output: **week3b** mounts them for the serving experiments.

In [ ]:
BASE = "Qwen/Qwen3-1.7B"
OUT = "/kaggle/working/week3a"
SPIDER = "/kaggle/working/data/spider_data"
MODELS = "/kaggle/working/models"
FP16 = f"{MODELS}/fp16"
LLAMA = "/kaggle/working/llama.cpp"
VARIANTS = ["awq_w4a16", "gptq_w4a16", "w8a8_int8"]
GGUF = {"gguf_q8_0": "Q8_0", "gguf_q4_k_m": "Q4_K_M"}
CPU_EVAL_LIMIT = 200  # llama.cpp on 4 CPU cores: ~5-10 s per question

import os, subprocess, json, glob, shutil
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for k in ["MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]:
    try:
        os.environ[k] = _s.get_secret(k)
    except Exception:
        print(f"secret {k} not set; that integration stays off")
os.makedirs(OUT, exist_ok=True)
STATUS = {}

def run(step, cmd, needs=()):
    """Run a shell step, stream its output, keep a per-step log, never raise.
    A failed step marks itself so that steps depending on it are skipped, not re-failed."""
    blocked = [n for n in needs if STATUS.get(n) != "ok"]
    if blocked:
        STATUS[step] = "skipped"
        print(f"--- skip {step}: needs {blocked}")
        return False
    print(f"--- {step}: {cmd}")
    with open(f"{OUT}/{step}.log", "w") as log:
        proc = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            if "it/s]" not in line and "s/it]" not in line:
                print(line, end="")
    STATUS[step] = "ok" if proc.wait() == 0 else "failed"
    print(f"--- {step}: {STATUS[step]}")
    return STATUS[step] == "ok"

In [ ]:
import glob, shutil, subprocess, tarfile
DEST = "/kaggle/working/repo"
marker = glob.glob("/kaggle/input/**/COMMIT", recursive=True)
if marker:  # code uploaded by scripts/kaggle_run.py
    commit = open(marker[0]).read().strip()
    tars = glob.glob("/kaggle/input/**/src.tar", recursive=True)
    trees = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
    os.makedirs(DEST, exist_ok=True)
    if tars:
        with tarfile.open(tars[0]) as t:
            t.extractall(DEST, filter="data")
    elif trees:  # Kaggle unpacks uploaded archives itself (into src/)
        shutil.copytree(os.path.dirname(trees[0]), DEST, dirs_exist_ok=True)
    else:
        raise RuntimeError("code dataset attached but neither src.tar nor pyproject.toml found")
else:
    !git clone -q {REPO} {DEST}
    commit = subprocess.getoutput(f"git -C {DEST} rev-parse --short HEAD")
open(f"{OUT}/COMMIT", "w").write(commit + "\n")
print("code commit:", commit)
%cd {DEST}
!pip install -q -e . -r requirements-gpu.txt 2>&1 | tail -3
# The image ships torchao 0.10; PEFT 0.20 raises on torchao < 0.16 when loading LoRA onto a plain
# (non-quantized) model, which broke the merge in run 3. Nothing here uses torchao.
!pip uninstall -y -q torchao

# Fail fast: a broken setup must not burn GPU hours on cells that can only fail.
check = subprocess.run(["python", "-c", "import quantize.compress, quantize.gguf, evaluation.perplexity, evaluation.generate, llmcompressor, vllm, peft"],
                       capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError("repo or serving deps not importable:\n" + check.stderr[-2000:])
print("repo and serving deps import OK")

## 1. Environment record

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv | tee {OUT}/gpu.txt
!pip freeze | grep -iE "^(vllm|torch|transformers|peft|llmcompressor|compressed-tensors)==" | tee {OUT}/versions.txt
!nproc | tee {OUT}/cpus.txt

## 2. Data, and the Week 1 model rebuilt as fp16 (the reference every variant is compared to)

In [ ]:
!python scripts/get_spider.py --out /kaggle/working/data
# Match on the weights file: the code dataset also holds a committed adapter_config.json without weights.
found = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
if found:
    run("merge", f"python -m finetune.merge --base {BASE} --adapter {os.path.dirname(found[0])} --out {FP16}")
else:
    STATUS["merge"] = "failed"
    print("no adapter attached: run with --attach models/week1_run3/sft/adapter=qwen-sql-week1-adapter")

## 3. Quantize (llm-compressor, one GPU)
Each variant is independent: one failing doesn't block the others.

In [ ]:
os.makedirs(f"{OUT}/quant", exist_ok=True)
for v in VARIANTS:
    if run(f"quant_{v}", f"CUDA_VISIBLE_DEVICES=0 python -m quantize.compress --model {FP16} --variant {v} "
           f"--spider-dir {SPIDER} --out {MODELS}/{v}", needs=["merge"]):
        shutil.copy(f"{MODELS}/{v}/quant_info.json", f"{OUT}/quant/{v}.json")

## 4. GGUF export (llama.cpp built from the pinned tag, CPU only)

In [ ]:
if run("gguf", f"python -m quantize.gguf --model {FP16} --out {MODELS}/gguf --llama-cpp {LLAMA}", needs=["merge"]):
    info = json.load(open(f"{MODELS}/gguf/gguf_info.json"))
    for key, qtype in GGUF.items():
        json.dump(info["files"][qtype] | {"llama_cpp_ref": info["llama_cpp_ref"]}, open(f"{OUT}/quant/{key}.json", "w"), indent=2)

## 5. Quality of every variant: Spider exec accuracy, perplexity, flips vs fp16
GGUF runs through vLLM's GGUF loader here (same engine and prompts as the rest); its CPU numbers come in step 6.

In [ ]:
def model_args(key):
    if key in GGUF:
        f = json.load(open(f"{OUT}/quant/{key}.json"))["file"]
        return f"--model {MODELS}/gguf/{f} --tokenizer {FP16} --hf-config-path {FP16}", ["gguf"]
    return f"--model {MODELS}/{key}", ["merge" if key == "fp16" else f"quant_{key}"]

for key in ["fp16", *VARIANTS, *GGUF]:
    args, needs = model_args(key)
    run(f"eval_{key}", f"CUDA_VISIBLE_DEVICES=0 python -m evaluation.generate {args} --spider-dir {SPIDER} "
        f"--out {OUT}/preds/{key}.jsonl", needs=needs)
    run(f"ppl_{key}", f"CUDA_VISIBLE_DEVICES=0 python -m evaluation.perplexity {args} --spider-dir {SPIDER} "
        f"--out {OUT}/ppl/{key}.json", needs=needs)
    if key != "fp16":
        run(f"compare_{key}", f"python -m evaluation.compare --spider-dir {SPIDER} --a {OUT}/preds/fp16.jsonl "
            f"--b {OUT}/preds/{key}.jsonl --out {OUT}/compare/{key}.json", needs=["eval_fp16", f"eval_{key}"])

## 6. Q4_K_M on llama.cpp CPU (the engine the cloud deployment uses), 200 dev questions
Same prompts through llama-server's OpenAI API. 4 slots (`-np 4`) x 2,048 tokens each covers the longest dev prompt (919) + output. Compared with fp16 on the same 200 questions.

In [ ]:
import time, signal
q4 = f"{MODELS}/gguf/" + json.load(open(f"{OUT}/quant/gguf_q4_k_m.json"))["file"] if STATUS.get("gguf") == "ok" else None
if q4:
    srv = subprocess.Popen(f"{LLAMA}/build/bin/llama-server -m {q4} --port 8080 -c 8192 -np 4 -t {os.cpu_count()} --alias qwen-sql",
                           shell=True, stdout=open(f"{OUT}/llama_server.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(120):  # model load on CPU: seconds
        if subprocess.run("curl -sf localhost:8080/health", shell=True).returncode == 0:
            break
        time.sleep(2)
    run("eval_gguf_q4_k_m_cpu", f"timeout 5400 python -m evaluation.remote --url http://localhost:8080 --tokenizer {FP16} "
        f"--spider-dir {SPIDER} --limit {CPU_EVAL_LIMIT} --concurrency 4 --out {OUT}/preds/gguf_q4_k_m_cpu.jsonl")
    os.killpg(srv.pid, signal.SIGTERM)
    run("compare_gguf_q4_k_m_cpu", f"python -m evaluation.compare --spider-dir {SPIDER} --limit {CPU_EVAL_LIMIT} "
        f"--a {OUT}/preds/fp16.jsonl --b {OUT}/preds/gguf_q4_k_m_cpu.jsonl --out {OUT}/compare/gguf_q4_k_m_cpu.json",
        needs=["eval_fp16", "eval_gguf_q4_k_m_cpu"])
else:
    STATUS["eval_gguf_q4_k_m_cpu"] = "skipped"

## 7. Quality table, model manifest for week3b, results zip

In [ ]:
run("variants_table", f"python -m bench.matrix --experiments {OUT}/none --quality {OUT} --out {OUT}/report")
present = [v for v in ["fp16", *VARIANTS] if os.path.exists(f"{MODELS}/{v}/config.json")]
json.dump({"variants": present, "gguf": os.listdir(f"{MODELS}/gguf") if os.path.isdir(f"{MODELS}/gguf") else []},
          open(f"{MODELS}/MANIFEST.json", "w"), indent=2)
print("models for week3b:", present)
json.dump(STATUS, open(f"{OUT}/status.json", "w"), indent=2)
print("step status:", STATUS)
shutil.make_archive("/kaggle/working/week3a_results", "zip", OUT)
for d in ["/kaggle/working/data", LLAMA]:  # keep kernel output small
    shutil.rmtree(d, ignore_errors=True)
print("download /kaggle/working/week3a_results.zip")